# Pasar Rau Sales Analysis
# Data Understanding

## Objective

The purpose of this notebook is to understand the structure, quality,
and limitations of the raw 2021 Pasar Rau sales dataset before performing
data cleaning or business analysis.

### 1. Questions

1. What does one row represent?
2. How many valid records and columns are available?
3. What are the expected data types?
4. Which fields contain missing or inconsistent values?
5. Are identifiers reliable?
6. Are there duplicate records?
7. Are financial fields internally consistent?
8. What data-quality issues need to be addressed before analysis?

### 2. Import libraries

In [24]:
from pathlib import Path

import pandas as pd
import numpy as np

In [25]:
## We're specifically trying to understand the dataset so we change the display settings without altering the underlying data.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

### 3. Define the raw file

In [26]:
RAW_DATA_PATH = Path("../data/raw/Penjualan2021-Pasarawu.csv")

RAW_DATA_PATH.exists()

True

### 4. CSV Handling
We already know something unusual about the source:
the header uses commas:

`Week,Month,Quart,DistID,...`

while actual data rows use semicolons:

`1;1;1;93047001;PT. FAJAR PANGAN LESTARI;...`

And the decimal separator is also a comma:

`11998,08`
`20149,2`
`221641,2`

So first, let's deliberately retrieve the header separately.

In [27]:
with RAW_DATA_PATH.open("r", encoding="utf-8-sig") as file:
    header_line = file.readline().strip()

columns = header_line.split(",")

columns

len(columns)

29

We're asking:
How many variables does the source claim to have?

Answer:
29.

### 5. Read everything as text first

In [28]:
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    sep=";",
    skiprows=1,
    names=columns,
    usecols=range(len(columns)),
    dtype=str,
    engine="python"
)

We're telling pandas:
"Don't be clever yet."

Especially because we have IDs like:
`367304001000328956`

If pandas decides that should become a floating-point number, we risk:
`3.6730400100032896e+17`

and possibly lose digits.
Identifiers aren't quantities.
You don't calculate:
`customer_id * 2`

So they generally belong as strings, not numeric values.

### 6. Remove the useless records after the real dataset
We'll detect hundreds of thousands of trailing delimiters instead of pretending they don't exist.

In [29]:
df_raw.shape

(328416, 29)

In [30]:
df_raw.tail()

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
328411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",,,,,,,,,,,,,,,,,,,,,,,,,,,,"
328412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",,,,,,,,,,,,,,,,,,,,,,,,,,,,"
328413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",,,,,,,,,,,,,,,,,,,,,,,,,,,,"
328414,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",,,,,,,,,,,,,,,,,,,,,,,,,,,,"
328415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",,,,,,,,,,,,,,,,,,,,,,,,,,,,"


`Do those rows contain actual business records?`

No.

So we're going to define a valid record based on something that should exist for every actual sales row.

In [31]:
df = df_raw[df_raw["Week"].notna()].copy()

In [32]:
df.shape

(10678, 29)

In [33]:
df.tail()

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
10673,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",67974514,RINSO MOLTO ROYAL GOLD (6+1)288X20ML,0,8,0,"1996,798003",96,102109,"34036,33","-323,35","3371,29",0,RINSO ROYAL20ML,SKU TARGET,"0,034",Tuesday,"37084,27,,,,,,,,,,,,,,,,,,,"
10674,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68693865,MOLTO ANTI KUSUT PURPLE (6+1) 216X20ML,0,3,0,"712,797149",36,154145,"25690,83","-244,06","2544,67",0,MOL KUSUT20,SKU TARGET,"0,026",Tuesday,"27991,44,,,,,,,,,,,,,,,,,,,"
10675,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68688878,RINSO MOLTO PERFUME ESSENCE PWD 126X42G,1,0,0,"5539,968",126,93355,93355,"-886,87","9246,81",0,RINSO PE44,SKU TARGET,"0,093",Tuesday,"101714,94,,,,,,,,,,,,,,,,,,,,"
10676,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68740643,WIPOL KARBOL CEMARA SACHET NEW 144X40ML,1,0,0,"5745,6",144,104727,104727,"-994,91","10373,21",0,WIPOL SST,SKU TARGET,"0,105",Tuesday,"114105,3,,,,,,,,,,,,,,,,,,,,"
10677,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68593141,DOVE SHP PERAWATAN RONTOK SX 480X9ML,0,10,0,"1324,8",120,338182,"84545,5","-803,18","8374,23",0,DOV RONTOKSST,SKU TARGET,"0,085",Tuesday,"92116,55,,,,,,,,,,,,,,,,,,,"


Now the bottom contains actual transactions rather than meaningless delimiter.

### 7. One source-specific fix before profiling
The final Net field has trailing commas from the malformed export, something like:

` 221641,2,,,,,,,,,,,,,,,,, `

We don't want to convert it to numeric yet.

We only want to remove export garbage, while preserving the original decimal comma.

In [34]:
df["Net"] = df["Net"].str.replace(r",+$", "", regex=True)

For example:

`221641,2,,,,,,,`

becomes:

`221641,2`

We haven't converted:

`221641,2`

into:

`221641.2`

yet.

Right now we're preserving the business value while removing obvious structural garbage from the malformed file.

### 8. Verify the dataset

In [35]:
df.shape

(10678, 29)

10,678 rows does NOT mean 10,678 transactions.

We're not allowed to make that conclusion yet.

In [36]:
df.head()

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
0,1,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021000360,06/01/21,367304001000328956,"AAS, IBU","RAU ATAS,PSR",67762152,RINSO MOLTO ROSE FRESH LIQ 144X40ML,2,0,0,"11998,08",288,105382,210764,-9272,"20149,2",0,RINSO MOLTO40,SKU TARGET,"0,211",Wednesday,"221641,2"
1,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,"AAS, IBU","RAU ATAS,PSR",67791434,RINSO MOLTO ROSE FRESH PWD 12X240G,1,0,0,2880,12,46364,46364,"-2372,43","4399,16",0,RINSO240,SKU TARGET,"0,046",Wednesday,"48390,73"
2,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,"AAS, IBU","RAU ATAS,PSR",67945877,WIPOL KARBOL CEMARA SACHET 144X40ML,1,0,0,"5745,6",144,104727,104727,"-5982,53","9874,44",0,WIPOL SST,SKU TARGET,"0,105",Wednesday,"108618,91"
3,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,"AAS, IBU","RAU ATAS,PSR",67974514,RINSO MOLTO ROYAL GOLD (6+1)288X20ML,0,9,0,"2246,4",108,"98181,99999","36818,25","-3491,84","3332,64",0,RINSO ROYAL20ML,SKU TARGET,"0,037",Wednesday,"36659,5"
4,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,"AAS, IBU","RAU ATAS,PSR",68504487,MOLTO PEWANGI BLUE 24X280ML,0,0,3,"831,24",3,91636,"11454,5","-85,91","1136,86",0,MOLTOBLU280,SKU TARGET,"0,011",Wednesday,"12505,45"


In [37]:
df.tail()

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
10673,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",67974514,RINSO MOLTO ROYAL GOLD (6+1)288X20ML,0,8,0,"1996,798003",96,102109,"34036,33","-323,35","3371,29",0,RINSO ROYAL20ML,SKU TARGET,"0,034",Tuesday,"37084,27"
10674,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68693865,MOLTO ANTI KUSUT PURPLE (6+1) 216X20ML,0,3,0,"712,797149",36,154145,"25690,83","-244,06","2544,67",0,MOL KUSUT20,SKU TARGET,"0,026",Tuesday,"27991,44"
10675,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68688878,RINSO MOLTO PERFUME ESSENCE PWD 126X42G,1,0,0,"5539,968",126,93355,93355,"-886,87","9246,81",0,RINSO PE44,SKU TARGET,"0,093",Tuesday,"101714,94"
10676,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68740643,WIPOL KARBOL CEMARA SACHET NEW 144X40ML,1,0,0,"5745,6",144,104727,104727,"-994,91","10373,21",0,WIPOL SST,SKU TARGET,"0,105",Tuesday,"114105,3"
10677,52,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",28/12/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",68593141,DOVE SHP PERAWATAN RONTOK SX 480X9ML,0,10,0,"1324,8",120,338182,"84545,5","-803,18","8374,23",0,DOV RONTOKSST,SKU TARGET,"0,085",Tuesday,"92116,55"


In [38]:
df.sample(10, random_state=42)

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
9084,43,10,4,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",27/10/2021,367304000400327767,SRI (NIK),"RAU,PSR",68599007,CLEAR SHP ICM UNISEX SX 480X1X9ML,1,0,0,"5299,2",480,338182,338182,"-20926,7","31725,53",0,CLEAR IC 9,SKU TARGET,"0,338",Wednesday,"348980,83"
3023,46,11,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",18/11/2021,367301000600329508,"BAHRUDIN,H (NIK)",TANGGUL RAU SETELAH TK.SIGIT,68740637,WIPOL KARBOL CEMARA REFILL NEW 24X460ML,2,0,0,"23979,84",48,187636,375272,"-4690,9","37058,11",0,WIP460,SKU TARGET,"0,375",Thursday,"407639,21"
8296,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,30,SAEFUL EFENDI,"2,1003E+11",22/12/2021,367304000400328195,RAHMAT (NIK),PS.RAU DEKAT OJI 087774091884,68716853,CITRA TS FRESH GLOW LLF 144X70G,0,0,8,"559,9944",8,320073,"17781,83","-35,56","1774,62",0,CITRA TSFRESH70,SKU TARGET,"0,018",Wednesday,"19520,89"
10359,39,9,3,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",28/09/2021,367304000400327448,YUDI (NIK),"RAU,PSR.BLOKM,08081314235239",67974518,RINSO MOLTO ROSE FRESH (6+1)288X20ML,0,0,6,"124,798003",6,"98181,99999","2045,46","-4,09","204,14",0,RINSO MOLT20ML,SKU TARGET,"0,002",Tuesday,"2245,51"
5464,40,10,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",10/06/2021,367304000400327787,ITA.TK (NIK),PS.RAUJL.CIKEPUH 085959999589,68493419,FAIR & LOVELY G&L MVIT FC FOAM 12X12X9G,2,0,0,2592,288,327273,654546,"-73996,46","58054,96",0,FAL FF 9,SKU TARGET,"0,655",Monday,"638604,5"
1084,37,9,3,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",15/09/2021,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",68607656,LIFEBUOY TS MILDCARE ARG 144X75G,0,0,6,"449,9928",6,345600,14400,"-136,8","1426,32",0,LIF TS MILD75,SKU TARGET,"0,014",Wednesday,"15689,52"
346,36,9,3,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,"2,10021E+11",09/08/2021,367304001000328956,"AAS, IBU (NIK)","RAU ATAS,PSR 087774324000",67927862,DOVE SHP TOTAL DAMAGE TRMT 480X9ML,0,20,0,2250,240,338182,169091,"-10145,46","15894,55",0,DOVE RSAKSST,SKU TARGET,"0,169",Wednesday,"174840,9"
6662,11,3,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021010450,17/03/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",68549339,MOLTO AIO PINK 360X10ML NEW,0,2,0,"263,99736",24,127636,"8509,07","-340,36","816,87",0,MOL PINK 10,SKU TARGET,"0,009",Wednesday,"8985,58"
6485,44,11,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002100882-02,11/05/2021,367304000400327475,KUSNI (NIK),"RAU,PSR 087771121009",67762152,RINSO MOLTO ROSE FRESH LIQ 144X40ML,0,0,-6,"-249,956001",-6,105382,"-4390,92",0,"-439,09",0,RINSO MOLTO40,SKU TARGET,"-0,004",Wednesday,"-4830,1"
2447,37,9,3,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",14/09/2021,367304000300329558,ALIONG (NIK),"RAU,PSR 081218681900",68651949,LIFEBUOY SHP PRWTN RBT RNTK SX 480X1X9ML,0,1,0,"132,48",12,163636,"4090,9","-8,18","408,27",0,0,0,"0,004",Tuesday,"4490,99"


### 9. Inspect column names

In [39]:
df.columns.tolist()

['Week',
 'Month',
 'Quart',
 'DistID',
 'DistName',
 'Salesman',
 'SalesmaneNama',
 'INVNumber',
 'INVDate',
 'Outlet',
 'Outlet Name',
 'Outlet address',
 'SKUCode',
 'Product Name',
 'Case',
 'Dozen',
 'Pieces',
 'Weight',
 'TotalQuantity(PCS)',
 'PriceCase',
 'GSV',
 'Discount',
 'Tax',
 'PERIOD',
 'NOTES 1',
 'NOTES 2',
 '1000000',
 'PJP',
 'Net']

In [40]:
len(df.columns)

29

In [41]:
for i, column in enumerate(df.columns, start=1):
    print(f"{i:>2}. {column}")

 1. Week
 2. Month
 3. Quart
 4. DistID
 5. DistName
 6. Salesman
 7. SalesmaneNama
 8. INVNumber
 9. INVDate
10. Outlet
11. Outlet Name
12. Outlet address
13. SKUCode
14. Product Name
15. Case
16. Dozen
17. Pieces
18. Weight
19. TotalQuantity(PCS)
20. PriceCase
21. GSV
22. Discount
23. Tax
24. PERIOD
25. NOTES 1
26. NOTES 2
27. 1000000
28. PJP
29. Net


Column `7. SalesmaneNama` probably contains a typo in the original schema.

Column `27. 1000000` probably means GSV in millions.

### 10. Inspect data types

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10678 entries, 0 to 10677
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Week                10678 non-null  str  
 1   Month               10678 non-null  str  
 2   Quart               10678 non-null  str  
 3   DistID              10678 non-null  str  
 4   DistName            10678 non-null  str  
 5   Salesman            10678 non-null  str  
 6   SalesmaneNama       10678 non-null  str  
 7   INVNumber           10678 non-null  str  
 8   INVDate             10678 non-null  str  
 9   Outlet              10678 non-null  str  
 10  Outlet Name         10678 non-null  str  
 11  Outlet address      10678 non-null  str  
 12  SKUCode             10678 non-null  str  
 13  Product Name        10678 non-null  str  
 14  Case                10678 non-null  str  
 15  Dozen               10678 non-null  str  
 16  Pieces              10678 non-null  str  
 17  Weig

### 11. Create our first profiling table

In [43]:
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100),
    "unique": df.nunique(dropna=True).values,
})

profile.sort_values("missing_pct", ascending=False)

,column,dtype,non_null,missing,missing_pct,unique
0,Week,str,10678,0,0.00,52
1,Month,str,10678,0,0.00,12
2,Quart,str,10678,0,0.00,4
3,DistID,str,10678,0,0.00,1
4,DistName,str,10678,0,0.00,1
5,Salesman,str,10678,0,0.00,5
6,SalesmaneNama,str,10678,0,0.00,7
7,INVNumber,str,10678,0,0.00,619
8,INVDate,str,10678,0,0.00,259
9,Outlet,str,10678,0,0.00,53


## Initial Observations

1. There are 540 SKU Codes and product names in the dataset
2. Multiple rows share invoice numbers, which is consistent with invoices containing multiple product line items. However, some invoice identifiers are corrupted by scientific notation, so invoice-level analysis requires additional validation.
3. GSV does not consistently equal TotalQuantity(PCS) × PriceCase. Because quantity is expressed in pieces while PriceCase appears to be a case-level price, these fields may use different units and require further investigation.
4. Net appears to equal GSV + Discount + Tax. Tax also appears to be approximately 10% of GSV after discount. These relationships should be validated across the full dataset.
5. Salesman ID doesn't seem to be an identifier for one specific salesman because ID 21 appears to be SAEFUL EFENDI and SUKMADEWI. And Salesman 30 appears to be SAEFUL EFENDI and DETI HANDAYANI.

#### Let's test some assumptions

First one there are 540 unique values in each column.

It does not yet prove that every SKU maps one-to-one with exactly one product name.

That becomes our next question:

`Does each SKU code correspond to exactly one product name?`

In [44]:
sku_mapping = (
    df.groupby("SKUCode")["Product Name"]
      .nunique()
      .sort_values(ascending=False)
)

sku_mapping.head(100)

SKUCode
20068201    1
20095361    1
20096041    1
20228327    1
20257721    1
20288827    1
21010428    1
21010442    1
21011149    1
21011154    1
21023189    1
21023192    1
21023193    1
21023197    1
21035423    1
21037727    1
21071998    1
21072126    1
21072789    1
21133325    1
21133328    1
21133369    1
21138171    1
21148071    1
21148088    1
21148089    1
21148091    1
21148092    1
21148093    1
21148094    1
21148096    1
21185944    1
21185972    1
21185974    1
60024416    1
60024446    1
62072023    1
62072024    1
62072025    1
62072030    1
67005966    1
67006105    1
67009810    1
67009847    1
67019477    1
67034313    1
67039480    1
67053685    1
67053687    1
67054374    1
67069761    1
67069763    1
67069765    1
67086153    1
67086155    1
67103391    1
67206111    1
67310658    1
67310660    1
67313235    1
67330745    1
67332056    1
67343319    1
67428446    1
67428631    1
67428633    1
67444019    1
67465879    1
67465880    1
67465881    1
67469524    

And the answer is yes. Every SKU Code belongs to a specific product name.

#### Now Let's Investigate Salesman ID

**Observation:** Salesman ID 21 is associated with both SAEFUL EFENDI
and SUKMADEWI.

**Hypothesis:** Salesman IDs may represent assignments or territories
that can be transferred between employees.

**Next validation:** Compare the date ranges associated with each
ID/name combination.

We're looking for something like this: 

`21 | SAEFUL EFENDI | Jan → May`

`21 | SUKMADEWI     | Jun → Dec`

In [45]:
inv_date_temp = pd.to_datetime(
    df["INVDate"],
    dayfirst=True,
    errors="coerce"
)

temp_salesman = df.assign(INVDate_temp=inv_date_temp)

salesman_mapping = (
    temp_salesman
    .groupby(["Salesman", "SalesmaneNama"])
    .agg(
        first_date=("INVDate_temp", "min"),
        last_date=("INVDate_temp", "max"),
        rows=("Salesman", "size")
    )
    .reset_index()
)

salesman_mapping

C:\Users\Thomas\AppData\Local\Temp\ipykernel_12036\1736469732.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  inv_date_temp = pd.to_datetime(


,Salesman,SalesmaneNama,first_date,last_date,rows
0,21,SAEFUL EFENDI,2021-01-04,2021-10-04,1824
1,21,SUKMADEWI,2021-01-09,2021-12-31,2399
2,30,DETI HANDAYANI,2021-02-10,2021-12-10,1212
3,30,SAEFUL EFENDI,2021-01-12,2021-12-31,708
4,30,ZUL RUSMANSYAH,2021-01-05,2021-12-08,2366
5,49,OB TELEORDER,2021-01-11,2021-10-02,189
6,51,ERTM FAJAR SWAT,2021-01-07,2021-12-31,237
7,61,ADANG,2021-01-11,2021-12-31,1743


There are 5 Salesman codes: 21, 30, 49, 51, and 61

It doesn't seem that the salesman code mapping change systematically over time because their periods are overlapping

The salesman codes might be mapped with a different rule.

## Data Quality and Relationship Validation

Initial profiling revealed several relationships and potential quality
issues that require further investigation:

- SKU-to-product mapping
- Invoice identifier reliability
- Salesperson identifier consistency
- Missing values
- Duplicate records
- Financial relationships

#### A. Missing values

In [46]:
missing_summary = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100
    })
    .sort_values("missing_pct", ascending=False)
)

missing_summary

,missing_count,missing_pct
Week,0,0.00
Month,0,0.00
Quart,0,0.00
DistID,0,0.00
DistName,0,0.00
Salesman,0,0.00
SalesmaneNama,0,0.00
INVNumber,0,0.00
INVDate,0,0.00
Outlet,0,0.00


#### B. Validate SKU mapping

In [50]:
sku_name_count = (
    df.groupby("SKUCode")["Product Name"]
      .nunique()
      .sort_values(ascending=False)
)

sku_name_count.head(20)

SKUCode
20068201    1
20095361    1
20096041    1
20228327    1
20257721    1
20288827    1
21010428    1
21010442    1
21011149    1
21011154    1
21023189    1
21023192    1
21023193    1
21023197    1
21035423    1
21037727    1
21071998    1
21072126    1
21072789    1
21133325    1
Name: Product Name, dtype: int64

In [51]:
(sku_name_count > 1).sum()

np.int64(0)

This tells us how many SKU codes map to multiple product names, which in this case = 0.

In [53]:
product_sku_count = (
    df.groupby("Product Name")["SKUCode"]
      .nunique()
      .sort_values(ascending=False)
)

product_sku_count.head(20)

Product Name
AXE BLACK POCKET FRAG 6X6X17ML             1
AXE DARK TMPTATION POCKET FRAG 6X6X17ML    1
AXE DEO BLACK 1X12X135ML                   1
AXE DEO BS DARK TEMPTATION 1X12X135ML      1
AXE DEO BS GOLD TEMPTATION 1X12X135ML      1
AXE DEO BS YOU COOL CHARGE 1X12X135ML      1
AXE DEO GOLD TEMP 1X12X135ML               1
BANGO MANIS 24X210ML                       1
BANGO MANIS HITAM GURIH 48X40ML            1
BANGO MANIS PROMO 48X(60+6ML)              1
BANGO MANIS RL 12X400ML                    1
BANGO MANIS RL 12X550ML                    1
BANGO MANIS RL 144X20ML                    1
BANGO MANIS RL 24X220ML                    1
BANGO MANIS RL 24X275ML                    1
BANGO MANIS RL 48X135ML                    1
BANGO MANIS RL 48X60ML                     1
BANGO POUCH FLATPACK 24X220ML              1
BUAVITA GUAVA WD 24X250ML                  1
BUAVITA MANGO WD 24X250ML                  1
Name: SKUCode, dtype: int64

In [54]:
(product_sku_count > 1).sum()

np.int64(0)

We reverse-checked if one product name map to multiple SKU codes? And the answer is there are none.

#### C. Investigate salesperson mappings

In [55]:
salesman_mapping

,Salesman,SalesmaneNama,first_date,last_date,rows
0,21,SAEFUL EFENDI,2021-01-04,2021-10-04,1824
1,21,SUKMADEWI,2021-01-09,2021-12-31,2399
2,30,DETI HANDAYANI,2021-02-10,2021-12-10,1212
3,30,SAEFUL EFENDI,2021-01-12,2021-12-31,708
4,30,ZUL RUSMANSYAH,2021-01-05,2021-12-08,2366
5,49,OB TELEORDER,2021-01-11,2021-10-02,189
6,51,ERTM FAJAR SWAT,2021-01-07,2021-12-31,237
7,61,ADANG,2021-01-11,2021-12-31,1743


#### D. Investigate the grain

We're going to investigate the data grain, does one row represents one product line within an invoice?

First, we pick a reliable-looking invoice—preferably one not written in scientific notation.

In [56]:
invoice_counts = df["INVNumber"].value_counts()

invoice_counts.head(20)

INVNumber
2,1003E+11        2797
2,10021E+11       2505
2,10061E+11       1627
2,10127E+11        104
21003000448-02      36
21006100157-02      33
21002100881-02      32
21003000243-02      30
21002100882-02      30
210030004940        29
210030000230        28
210030003190        25
21003000463-02      25
210030007220        23
210021010450        22
210021017880        22
21003001760         21
210030001210        21
21002103206         21
210030003810        21
Name: count, dtype: int64

In [57]:
df[df["INVNumber"] == "210030004940"]

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
586,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67714602,VIXAL PEMB PORS BIRU BTL 24X190ML,0,0,6,"1224,96",6,91636,22909,"-217,64","2269,14",0,VIX 175,SKU TARGET,"0,023",Wednesday,"24960,5"
587,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67791623,MOLTO EDP PURPLE SC 360X10ML,0,10,0,"1185,598814",120,127636,"42545,33","-2089,82","4045,55",0,MOL PURPL 10,SKU TARGET,"0,043",Wednesday,"44501,6"
588,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67791625,MOLTO EDP BLUE FRENCH LILY SC 360X10ML,0,10,0,"1185,598814",120,127636,"42545,33","-404,18","4214,11",0,0,0,"0,043",Wednesday,"46355,26"
589,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67791626,MOLTO EDP PINK ROSE SC 360X10ML,0,10,0,"1185,598814",120,127636,"42545,33","-404,18","4214,11",0,0,0,"0,043",Wednesday,"46355,26"
590,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67833343,MOLTO PURE SACHET SC 360X10ML,0,10,0,"1185,598814",120,127636,"42545,33","-404,18","4214,11",0,0,0,"0,043",Wednesday,"46355,26"
591,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67907162,LIFEBUOY SHP PRAWTN RMBT RNTK 480X1X9ML,0,5,0,"562,5",60,163636,"20454,5","-194,32","2026,02",0,0,0,"0,020",Wednesday,"22286,2"
592,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67909116,LUX TS SOFT ROSE 144X75G,0,0,6,"449,9928",6,366545,"15272,71","-145,09","1512,76",0,LUX TS PINK75,SKU TARGET,"0,015",Wednesday,"16640,38"
593,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67909118,LUX TS VELVET JASMINE 144X75G,0,0,6,"431,993088",6,366545,"15272,71","-145,09","1512,76",0,LUX TS VELVET75,SKU TARGET,"0,015",Wednesday,"16640,38"
594,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67909126,LUX TS LILY FRESH 144X75G,0,0,6,"449,9928",6,366545,"15272,71","-145,09","1512,76",0,0,0,"0,015",Wednesday,"16640,38"
595,10,3,1,93047001,PT. FAJAR PANGAN LESTARI,30,ZUL RUSMANSYAH,210030004940,10/03/21,367304000400327538,ABDUR ROJAK (NIK),"RAU,PSR 083871008203",67927854,DOVE SHP DANDRUFF CARE 480X9ML,0,5,0,"562,5",60,338182,"42272,75","-401,59","4187,11",0,DOV KETOMBESST,SKU TARGET,"0,042",Wednesday,"46058,27"


In [59]:
df.loc[
    df["INVNumber"] == "210030004940",
    [
        "INVNumber",
        "INVDate",
        "Outlet Name",
        "SKUCode",
        "Product Name",
        "TotalQuantity(PCS)",
        "GSV",
        "Net"
    ]
]

,INVNumber,INVDate,Outlet Name,SKUCode,Product Name,TotalQuantity(PCS),GSV,Net
586,210030004940,10/03/21,ABDUR ROJAK (NIK),67714602,VIXAL PEMB PORS BIRU BTL 24X190ML,6,22909,"24960,5"
587,210030004940,10/03/21,ABDUR ROJAK (NIK),67791623,MOLTO EDP PURPLE SC 360X10ML,120,"42545,33","44501,6"
588,210030004940,10/03/21,ABDUR ROJAK (NIK),67791625,MOLTO EDP BLUE FRENCH LILY SC 360X10ML,120,"42545,33","46355,26"
589,210030004940,10/03/21,ABDUR ROJAK (NIK),67791626,MOLTO EDP PINK ROSE SC 360X10ML,120,"42545,33","46355,26"
590,210030004940,10/03/21,ABDUR ROJAK (NIK),67833343,MOLTO PURE SACHET SC 360X10ML,120,"42545,33","46355,26"
591,210030004940,10/03/21,ABDUR ROJAK (NIK),67907162,LIFEBUOY SHP PRAWTN RMBT RNTK 480X1X9ML,60,"20454,5","22286,2"
592,210030004940,10/03/21,ABDUR ROJAK (NIK),67909116,LUX TS SOFT ROSE 144X75G,6,"15272,71","16640,38"
593,210030004940,10/03/21,ABDUR ROJAK (NIK),67909118,LUX TS VELVET JASMINE 144X75G,6,"15272,71","16640,38"
594,210030004940,10/03/21,ABDUR ROJAK (NIK),67909126,LUX TS LILY FRESH 144X75G,6,"15272,71","16640,38"
595,210030004940,10/03/21,ABDUR ROJAK (NIK),67927854,DOVE SHP DANDRUFF CARE 480X9ML,60,"42272,75","46058,27"


These columns stays constant between rows: 

`INVNumber`	

`INVDate`

`Outlet Name`

These columns changes between rows in the same invoice:
`SKUCode`

`Product Name`

`TotalQuantity(PCS)`	

`GSV`	

`Net`

Now we've got strong evidence that:
`one row represents one product line within an invoice.`

### Validation Findings

1.  Observation: Sales Code doesn't represent a certain range of period between salesperon name.
    Hyphotesis: Maybe the sales code represent an area where the salespersons works or something else
2.  Observation: After investigating, we've found that a row represents one product line within an invoice
3.  Observation: We investigated SKU and there are no SKU codes that maps to multiple product names and we reverse checked, and the answer is still none (there are no products linked to multiple SKUs)
4.  Observation: There are no missing values left in the dataframe.

## 1. Check blank-like values

#### Blank and Placeholder Value Check

Although no standard missing values were detected, text-based datasets may
contain blank strings or placeholder values that are not automatically
recognized as missing by pandas.

In [60]:
blank_summary = {}

for col in df.columns:
    series = df[col].astype(str).str.strip()

    blank_summary[col] = {
        "blank": (series == "").sum(),
        "NA_text": series.str.upper().isin(["N/A", "NA", "NULL", "NONE", "-"]).sum()
    }

blank_summary = pd.DataFrame(blank_summary).T

blank_summary[
    (blank_summary["blank"] > 0) |
    (blank_summary["NA_text"] > 0)
].sort_values(["blank", "NA_text"], ascending=False)

,blank,NA_text


This tells us that there are no blank strings or placeholder values

### 2. Duplicate investigation

#### Duplicate Record Investigation

Duplicate rows must be investigated before removal because identical-looking
records may represent either data-entry duplication or legitimate repeated
transaction lines.

In [61]:
duplicate_count = df.duplicated().sum()
duplicate_count

np.int64(21)

Turns out there are 21 rows of duplicated pairs. This either mean data-entry duplication or legitimate repeated transaction lines.

In [62]:
duplicates = df[df.duplicated(keep=False)].sort_values(
    ["INVDate", "INVNumber", "Outlet Name", "SKUCode"]
)

duplicates

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
2533,40,10,4,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",10/05/2021,367304000300329558,ALIONG (NIK),"RAU,PSR 081218681900",68571429,BANGO MANIS PROMO 48X(60+6ML),50,0,0,221520,2400,104727,5236350,"-333948,22","490240,18",0,BANGO 60,SKU TARGET,"5,236",Tuesday,"5392641,96"
2534,40,10,4,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",10/05/2021,367304000300329558,ALIONG (NIK),"RAU,PSR 081218681900",68571429,BANGO MANIS PROMO 48X(60+6ML),50,0,0,221520,2400,104727,5236350,"-333948,22","490240,18",0,BANGO 60,SKU TARGET,"5,236",Tuesday,"5392641,96"
2065,49,12,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",12/08/2021,367304000400327430,ALAMI (NIK),PS.RAURAU BLOK B1 NO.1 081932888788,68607663,LIFEBUOY TS LEMONFRESH ARG 36X(4X60G),1,0,0,8640,36,244800,244800,"-3549,6","24125,04",0,LIF MP4 LEMON,SKU TARGET,"0,245",Wednesday,"265375,44"
2098,49,12,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",12/08/2021,367304000400327430,ALAMI (NIK),PS.RAURAU BLOK B1 NO.1 081932888788,68607663,LIFEBUOY TS LEMONFRESH ARG 36X(4X60G),1,0,0,8640,36,244800,244800,"-3549,6","24125,04",0,LIF MP4 LEMON,SKU TARGET,"0,245",Wednesday,"265375,44"
2062,49,12,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",12/08/2021,367304000400327430,ALAMI (NIK),PS.RAURAU BLOK B1 NO.1 081932888788,68607665,LIFEBUOY TS MILDCARE ARG 36X(4X60G),1,0,0,8640,36,244800,244800,"-3549,6","24125,04",0,LIF MP4 MILD,SKU TARGET,"0,245",Wednesday,"265375,44"
2105,49,12,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",12/08/2021,367304000400327430,ALAMI (NIK),PS.RAURAU BLOK B1 NO.1 081932888788,68607665,LIFEBUOY TS MILDCARE ARG 36X(4X60G),1,0,0,8640,36,244800,244800,"-3549,6","24125,04",0,LIF MP4 MILD,SKU TARGET,"0,245",Wednesday,"265375,44"
6986,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67005966,LIFEBUOY TS MILDCARE 36X(4X60G),0,0,-6,"-1439,99424",-6,244800,-40800,0,-4080,0,LIF MP4 MILD,SKU TARGET,"-0,041",Wednesday,-44880
7002,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67005966,LIFEBUOY TS MILDCARE 36X(4X60G),0,0,-6,"-1439,99424",-6,244800,-40800,0,-4080,0,LIF MP4 MILD,SKU TARGET,"-0,041",Wednesday,-44880
6993,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67034313,LIFEBUOY TS LEMONFRESH 36X(4X60G),0,0,-6,"-1439,99424",-6,244800,-40800,0,-4080,0,LIF MP4 LEMON,SKU TARGET,"-0,041",Wednesday,-44880
6999,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67034313,LIFEBUOY TS LEMONFRESH 36X(4X60G),0,0,-6,"-1439,99424",-6,244800,-40800,0,-4080,0,LIF MP4 LEMON,SKU TARGET,"-0,041",Wednesday,-44880


We're not going to remove these duplicates yet.  

It could be:
- duplicated export row
- legitimate repeated purchase
- repeated return
- invoice corruption causing records to look identical

We're going to summarize duplicate groups

In [63]:
duplicate_groups = (
    duplicates
    .groupby(list(df.columns), dropna=False)
    .size()
    .reset_index(name="duplicate_count")
    .sort_values("duplicate_count", ascending=False)
)

duplicate_groups.head(20)

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,Outlet Name,Outlet address,SKUCode,Product Name,Case,Dozen,Pieces,Weight,TotalQuantity(PCS),PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net,duplicate_count
0,30,7,3,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,"2,10021E+11",29/07/2021,367304000400327721,YANTI (NIK),"RAU,PSR",68386572,SUNLIGHT LIME NEW POUCH 72X105ML,100,0,0,783000,7200,106560,10656000,"-1099645,92","955635,41",0,SUNL 105,SKU TARGET,"10,656",Wednesday,"10511989,49",2
1,40,10,4,93047001,PT. FAJAR PANGAN LESTARI,30,DETI HANDAYANI,"2,1003E+11",10/05/2021,367304000300329558,ALIONG (NIK),"RAU,PSR 081218681900",68571429,BANGO MANIS PROMO 48X(60+6ML),50,0,0,221520,2400,104727,5236350,"-333948,22","490240,18",0,BANGO 60,SKU TARGET,"5,236",Tuesday,"5392641,96",2
2,49,12,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",12/08/2021,367304000400327430,ALAMI (NIK),PS.RAURAU BLOK B1 NO.1 081932888788,68607663,LIFEBUOY TS LEMONFRESH ARG 36X(4X60G),1,0,0,8640,36,244800,244800,"-3549,6","24125,04",0,LIF MP4 LEMON,SKU TARGET,"0,245",Wednesday,"265375,44",2
3,49,12,4,93047001,PT. FAJAR PANGAN LESTARI,61,ADANG,"2,10061E+11",12/08/2021,367304000400327430,ALAMI (NIK),PS.RAURAU BLOK B1 NO.1 081932888788,68607665,LIFEBUOY TS MILDCARE ARG 36X(4X60G),1,0,0,8640,36,244800,244800,"-3549,6","24125,04",0,LIF MP4 MILD,SKU TARGET,"0,245",Wednesday,"265375,44",2
4,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67005966,LIFEBUOY TS MILDCARE 36X(4X60G),0,0,-6,"-1439,99424",-6,244800,-40800,0,-4080,0,LIF MP4 MILD,SKU TARGET,"-0,041",Wednesday,-44880,2
5,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67034313,LIFEBUOY TS LEMONFRESH 36X(4X60G),0,0,-6,"-1439,99424",-6,244800,-40800,0,-4080,0,LIF MP4 LEMON,SKU TARGET,"-0,041",Wednesday,-44880,2
6,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",67907149,LIFEBUOY SHP ANTI KETOMBE 480X1X9ML,0,-2,0,-225,-24,163636,"-8181,8",0,"-818,18",0,LIF AD 9,SKU TARGET,"-0,008",Wednesday,"-8999,98",2
7,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",68492300,RINSO MOLTO ROSE FRESH PWD 12X460G,0,0,-2,"-919,99632",-2,91800,-15300,0,-1530,0,RINSO460,SKU TARGET,"-0,015",Wednesday,-16830,2
8,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",68492302,RINSO MOLTO PERFUME ESSENCE PWD 12X460G,0,0,-2,"-919,99632",-2,91800,-15300,0,-1530,0,RINSO460,SKU TARGET,"-0,015",Wednesday,-16830,2
9,51,12,4,93047001,PT. FAJAR PANGAN LESTARI,21,SUKMADEWI,21002101034-02,25/12/2021,367304000500327653,MAHMUDI (NIK),"LOS ATAS.RAU,087820679003",68519758,RINSO ANTI NODA CLASSIC PWD 12X460G,0,0,-2,"-919,99632",-2,91800,-15300,0,-1530,0,RINSO460,SKU TARGET,"-0,015",Wednesday,-16830,2


Inspections:
- Most of the duplicates are returns/transaction with negative values.
- There are only 4 groups of duplicates with positive values.
- The duplicate groups have varied products
- Many of the duplicate groups occured in 25/12/2021 and happened in 4 different outlets in 4 invoices
- Every groups occur exactly 2 times.

### 3. Financial reconciliation

In [64]:
def parse_indonesian_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "":
        return np.nan

    # If both separators exist, assume:
    # dot = thousands separator
    # comma = decimal separator
    if "." in value and "," in value:
        value = value.replace(".", "").replace(",", ".")

    # If only comma exists, treat comma as decimal separator
    elif "," in value:
        value = value.replace(",", ".")

    return pd.to_numeric(value, errors="coerce")

In [65]:
financial_cols = ["GSV", "Discount", "Tax", "Net"]

financial_temp = df[financial_cols].apply(
    lambda col: col.map(parse_indonesian_number)
)

financial_temp.head()

,GSV,Discount,Tax,Net
0,"210,764.00","-9,272.00","20,149.20","221,641.20"
1,"46,364.00","-2,372.43","4,399.16","48,390.73"
2,"104,727.00","-5,982.53","9,874.44","108,618.91"
3,"36,818.25","-3,491.84","3,332.64","36,659.50"
4,"11,454.50",-85.91,"1,136.86","12,505.45"


In [66]:
financial_temp.isna().sum()

GSV         0
Discount    0
Tax         0
Net         0
dtype: int64

The parser successfully convert all expected financial values

### 4. Test Net

In [67]:
financial_check = financial_temp.copy()

financial_check["calculated_net"] = (
    financial_check["GSV"]
    + financial_check["Discount"]
    + financial_check["Tax"]
)

financial_check["net_difference"] = (
    financial_check["Net"]
    - financial_check["calculated_net"]
)

In [68]:
financial_check["net_difference"].describe()

count   10,678.00
mean         0.04
std          0.14
min         -0.81
25%          0.00
50%          0.00
75%          0.00
max          0.81
Name: net_difference, dtype: float64

In [69]:
financial_check["net_difference"].abs().sort_values(ascending=False).head(20)

9896   0.81
5827   0.81
2870   0.81
8737   0.81
4495   0.81
5991   0.81
8738   0.81
4494   0.81
4118   0.81
7516   0.81
4928   0.81
8758   0.81
8845   0.81
8432   0.81
7542   0.81
2706   0.81
8571   0.81
8470   0.81
3432   0.81
3397   0.81
Name: net_difference, dtype: float64

How far does reported Net differ from calculated Net?

In [81]:
tolerance = 0.1

net_mismatch = financial_check[
    financial_check["net_difference"].abs() > tolerance
]

print(len(net_mismatch))
net_mismatch.head(20)

1166


,GSV,Discount,Tax,Net,calculated_net,net_difference,after_discount,expected_tax_10pct,tax_difference,discount_rate
3,"36,818.25","-3,491.84","3,332.64","36,659.50","36,659.05",0.45,"33,326.41","3,332.64",-0.00,0.09
17,"84,545.50",0.00,"8,454.55","93,000.50","93,000.05",0.45,"84,545.50","8,454.55",-0.00,0.00
95,"23,454.50","-1,572.62","2,188.18","24,070.60","24,070.06",0.54,"21,881.88","2,188.19",-0.01,0.07
105,"-23,454.50","1,572.62","-2,188.18","-24,070.60","-24,070.06",-0.54,"-21,881.88","-2,188.19",0.01,0.07
107,"126,818.25","-5,072.73","12,174.55","133,920.70","133,920.07",0.63,"121,745.52","12,174.55",-0.00,0.04
108,"42,272.75","-1,690.91","4,058.18","44,640.20","44,640.02",0.18,"40,581.84","4,058.18",-0.00,0.04
110,"42,272.75",0.00,"4,227.27","46,500.20","46,500.02",0.18,"42,272.75","4,227.28",-0.01,0.00
117,"13,636.38",-818.18,"1,281.82","14,100.20","14,100.02",0.18,"12,818.20","1,281.82",0.00,0.06
142,"36,818.25","-3,491.84","3,332.64","36,659.50","36,659.05",0.45,"33,326.41","3,332.64",-0.00,0.09
143,"36,818.25","-3,491.84","3,332.64","36,659.50","36,659.05",0.45,"33,326.41","3,332.64",-0.00,0.09


Most of the Net differences are very tiny, it might be because of rounding, so it's not a big deal. 

### 5. Test the 10% tax hypothesis

In [72]:
financial_check["after_discount"] = (
    financial_check["GSV"]
    + financial_check["Discount"]
)

In [73]:
financial_check["expected_tax_10pct"] = (
    financial_check["after_discount"] * 0.10
)

financial_check["tax_difference"] = (
    financial_check["Tax"]
    - financial_check["expected_tax_10pct"]
)

In [74]:
financial_check["tax_difference"].describe()

count   10,678.00
mean        -0.00
std          0.00
min         -0.01
25%         -0.00
50%          0.00
75%          0.00
max          0.05
Name: tax_difference, dtype: float64

In [75]:
financial_check["tax_difference"].abs().sort_values(ascending=False).head(20)

818     0.05
746     0.02
814     0.01
354     0.01
8627    0.01
1320    0.01
1401    0.01
10099   0.01
10116   0.01
3317    0.01
3323    0.01
9843    0.01
2381    0.01
1494    0.01
3311    0.01
7257    0.01
7883    0.01
5386    0.01
9611    0.01
780     0.01
Name: tax_difference, dtype: float64

In [76]:
tax_mismatch = financial_check[
    financial_check["tax_difference"].abs() > tolerance
]

len(tax_mismatch)

0


There are no mismatch of the tax calculations.

Now we know that the tax is 10% of the GSV + Discount.

### 6. Add one more metric: discount rate

In [77]:
financial_check["discount_rate"] = (
    financial_check["Discount"].abs()
    / financial_check["GSV"].abs()
)

In [78]:
financial_check["discount_rate"].describe()

c:\Users\Thomas\Desktop\Resume & CV\portfolio\pasar-rau-sales-analysis\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count   10,673.00
mean          inf
std           NaN
min          0.00
25%          0.00
50%          0.01
75%          0.04
max           inf
Name: discount_rate, dtype: float64

In [79]:
rates = financial_check["discount_rate"]
gsv = financial_temp["GSV"]
discount = financial_temp["Discount"]

print("Discount-rate summary:")
display(rates.describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

# Flag rates above 20%, plus undefined/infinite rates caused by zero GSV.
unusual_mask = rates.gt(0.20) | ~np.isfinite(rates)
unusual_idx = rates.index[unusual_mask]

print(f"Rows with discount rate > 20% or undefined: {len(unusual_idx)}")
print(f"Rows with discount rate > 100%: {rates.gt(1).sum()}")
print(f"Rows with zero GSV: {gsv.eq(0).sum()}")

source_cols = [
    "INVNumber", "INVDate", "Outlet Name", "SKUCode", "Product Name",
    "Case", "Dozen", "Pieces", "TotalQuantity(PCS)",
    "GSV", "Discount", "Tax", "Net"
]

unusual_rows = df.loc[unusual_idx, source_cols].copy()
unusual_rows["GSV_num"] = gsv.loc[unusual_idx]
unusual_rows["Discount_num"] = discount.loc[unusual_idx]
unusual_rows["discount_rate"] = rates.loc[unusual_idx]
unusual_rows["after_discount"] = (
    gsv.loc[unusual_idx] + discount.loc[unusual_idx]
)

# Distinguish denominator problems from discounts larger than GSV.
unusual_rows["diagnostic"] = np.select(
    [
        unusual_rows["GSV_num"].eq(0),
        unusual_rows["Discount_num"].abs() > unusual_rows["GSV_num"].abs(),
        unusual_rows["discount_rate"].gt(0.20),
    ],
    [
        "Zero GSV: rate is undefined or infinite",
        "Discount magnitude exceeds GSV",
        "High rate; inspect transaction context",
    ],
    default="",
)

display(
    unusual_rows.sort_values("discount_rate", ascending=False, na_position="first")
)

Discount-rate summary:


c:\Users\Thomas\Desktop\Resume & CV\portfolio\pasar-rau-sales-analysis\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\Thomas\Desktop\Resume & CV\portfolio\pasar-rau-sales-analysis\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count   10,673.00
mean          inf
std           NaN
min          0.00
50%          0.01
90%          0.07
95%          0.09
99%           NaN
max           inf
Name: discount_rate, dtype: float64

Rows with discount rate > 20% or undefined: 176
Rows with discount rate > 100%: 158
Rows with zero GSV: 163


,INVNumber,INVDate,Outlet Name,SKUCode,Product Name,Case,Dozen,Pieces,TotalQuantity(PCS),GSV,Discount,Tax,Net,GSV_num,Discount_num,discount_rate,after_discount,diagnostic
3365,21003000040-02,02/02/21,FUADI (NIK),67034313,LIFEBUOY TS LEMONFRESH 36X(4X60G),0,0,0,0,0,0,"0,01","0,1",0.00,0.00,NaN,0.00,Zero GSV: rate is undefined or infinite
3367,21003000040-02,02/02/21,FUADI (NIK),67006105,LIFEBUOY TS TOTAL10 36X(4X60G),0,0,0,0,0,0,"0,01","0,1",0.00,0.00,NaN,0.00,Zero GSV: rate is undefined or infinite
7754,210127000020,07/01/21,"NONG,H",G0001063,Paket Donasi,0,0,1,1,0,0,0,0,0.00,0.00,NaN,0.00,Zero GSV: rate is undefined or infinite
7832,210127001370,09/02/21,QORNI (NIK),G0001063,Paket Donasi,0,0,1,1,0,0,0,0,0.00,0.00,NaN,0.00,Zero GSV: rate is undefined or infinite
7840,210127002930,19/03/2021,QORNI (NIK),G0001063,Paket Donasi,0,0,1,1,0,0,0,0,0.00,0.00,NaN,0.00,Zero GSV: rate is undefined or infinite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4076,"2,10021E+11",06/02/2021,"HANIF,TK (NIK)",67974518,RINSO MOLTO ROSE FRESH (6+1)288X20ML,1,0,0,288,98182,"-20615,34","7756,66","85323,32","98,182.00","-20,615.34",0.21,"77,566.66",High rate; inspect transaction context
4077,"2,10021E+11",06/02/2021,"HANIF,TK (NIK)",68144275,RINSO ANTI NODA CLASSIC PWD 126X44G,0,0,6,6,"4390,9","-921,96","346,89","3815,83","4,390.90",-921.96,0.21,"3,468.94",High rate; inspect transaction context
4078,"2,10021E+11",06/02/2021,"HANIF,TK (NIK)",68410982,ROYCO FDS BEEF RL 576X9G,0,1,0,12,"4090,92","-858,97","323,19","3555,14","4,090.92",-858.97,0.21,"3,231.95",High rate; inspect transaction context
7785,21012700023-02,22/02/2021,"NONG,H (NIK)",68159714,JAWARA SAUS HOT POUCH 24X120ML,0,0,-3,-3,"-17443,63","3488,73","-1395,49","-15350,39","-17,443.63","3,488.73",0.20,"-13,954.90",High rate; inspect transaction context


In [80]:
zero_gsv_mask = financial_temp["GSV"].eq(0)
raw_gsv_zero = df.loc[zero_gsv_mask, "GSV"].astype(str).str.strip()

print(f"Rows parsed as zero GSV: {zero_gsv_mask.sum()}")
print("\nOriginal GSV values for those rows:")
display(raw_gsv_zero.value_counts().rename_axis("raw GSV").to_frame("rows"))

# Flag source strings that do not look like zero written with optional decimal zeros.
canonical_zero = raw_gsv_zero.str.fullmatch(r"[+-]?0+(?:[,.]0+)?")
suspicious_raw_values = raw_gsv_zero[~canonical_zero]

print(f"\nZero-parsed rows with non-zero-looking source text: {len(suspicious_raw_values)}")
if not suspicious_raw_values.empty:
    display(
        df.loc[suspicious_raw_values.index, [
            "INVNumber", "INVDate", "Outlet Name", "SKUCode",
            "Product Name", "GSV"
        ]]
    )
else:
    print("None: every parsed zero corresponds to a zero-looking value in the original GSV field.")

Rows parsed as zero GSV: 163

Original GSV values for those rows:


,rows
raw GSV,
0,163



Zero-parsed rows with non-zero-looking source text: 0
None: every parsed zero corresponds to a zero-looking value in the original GSV field.


Findings: 

There are some 0 values on GSV which causes the discount rates to be unusual.

## Data Quality Summary

### Reliable Fields
- The dataset covers a complete reporting period for 2021 and contains 10,678 valid record lines.
- All valid transaction rows contain parseable transaction dates covering 4 January through 31 December 2021.
- Outlet identifiers are preserved in the raw source but require mapping validation. Invoice identifiers are only partially preserved, with many values affected by scientific notation and therefore unsuitable for unrestricted invoice-level analysis.
- Net reconciles to GSV + Discount + Tax within less than Rp1 across the dataset, indicating only immaterial rounding differences. Tax reconciles to approximately 10% of GSV after discount within the selected tolerance.

### Fields Requiring Caution
- `GSV`, `Discount`, `Tax`, and `Net` require locale-aware parsing because the source uses inconsistent number formats.
- `INVNumber` is partially corrupted by scientific-notation conversion and should not be used blindly for transaction counting.
- `Outlet Name` and `Salesman` mappings need validation because code-to-name associations appear to change over time.
- `PriceCase`, `Dozen`, `Pieces`, and `TotalQuantity(PCS)` may not share the same unit conventions and should be verified before business interpretation.

### Data Quality Issues
- Some rows contain zero `GSV`, which makes discount-rate calculations undefined or misleading.
- Negative quantities and negative financial values indicate return or reversal activity that should be separated from normal sales.
- Duplicate records exist and require investigation before removing or aggregating them.
- Mixed formatting and inconsistent numeric representation are the main sources of parsing error in the financial columns.

### Confirmed Relationships
- Net sales are approximately equal to `GSV + Discount + Tax`, confirming the expected reconciliation logic.
- Tax appears to be around 10% of the adjusted value base, consistent with a tax-on-sale structure.
- The transaction dates are complete and aligned across the source records.

### Unresolved Questions
- How should product quantity fields be normalized when `Case`, `Dozen`, `Pieces`, and `TotalQuantity(PCS)` use different measurement units?
- Are the zero-GSV / unusually high discount rows genuine business transactions or data-entry anomalies?
- How should outlet-name and salesman-code reassignments be treated over time?
- Should returns, reversals, and duplicates be excluded from the main sales analysis or analyzed as a separate business stream?